In [181]:
import polars as pl


solar = pl.read_csv(r"c:\Users\HanhTa\Downloads\solar tech avail.csv")
wind = pl.read_csv(r"C:\Users\HanhTa\Downloads\techavail wind.csv")
energy_weather_load1 = pl.read_csv(r"c:\Users\HanhTa\Downloads\ewload1.csv")
energy_weather_load2 = pl.read_csv(r"c:\Users\HanhTa\Downloads\ew_load13.csv")
energy_weather_load3 = pl.read_csv(r"c:\Users\HanhTa\Downloads\ew_load35.csv")
energy_weather_load4 = pl.read_csv(r"c:\Users\HanhTa\Downloads\ew_load57.csv")
energy_weather_load5 = pl.read_csv(r"c:\Users\HanhTa\Downloads\ew_load79.csv")



In [182]:
df = pl.concat([energy_weather_load1, energy_weather_load2, energy_weather_load3, energy_weather_load4, energy_weather_load5], how="vertical")

display(df)

time,data,issued
str,f64,str
"""2026-01-01 00:00:00+00""",39975.528491,"""2025-12-31 09:00:00+00"""
"""2026-01-01 00:15:00+00""",39478.928456,"""2025-12-31 09:00:00+00"""
"""2026-01-01 00:30:00+00""",39161.142551,"""2025-12-31 09:00:00+00"""
"""2026-01-01 00:45:00+00""",38936.679901,"""2025-12-31 09:00:00+00"""
"""2026-01-01 01:00:00+00""",38782.980144,"""2025-12-31 09:00:00+00"""
…,…,…
"""2026-08-31 22:45:00+00""",48236.690756,"""2026-08-31 08:00:00+00"""
"""2026-08-31 23:00:00+00""",47884.138047,"""2026-08-31 08:00:00+00"""
"""2026-08-31 23:15:00+00""",47727.638161,"""2026-08-31 08:00:00+00"""


In [183]:
df = df.with_columns(
    pl.col(["time", "issued"])
    .str.to_datetime("%Y-%m-%d %H:%M:%S%#z")
    .dt.convert_time_zone("UTC")
)

In [184]:
display(df)

time,data,issued
"datetime[μs, UTC]",f64,"datetime[μs, UTC]"
2026-01-01 00:00:00 UTC,39975.528491,2025-12-31 09:00:00 UTC
2026-01-01 00:15:00 UTC,39478.928456,2025-12-31 09:00:00 UTC
2026-01-01 00:30:00 UTC,39161.142551,2025-12-31 09:00:00 UTC
2026-01-01 00:45:00 UTC,38936.679901,2025-12-31 09:00:00 UTC
2026-01-01 01:00:00 UTC,38782.980144,2025-12-31 09:00:00 UTC
…,…,…
2026-08-31 22:45:00 UTC,48236.690756,2026-08-31 08:00:00 UTC
2026-08-31 23:00:00 UTC,47884.138047,2026-08-31 08:00:00 UTC
2026-08-31 23:15:00 UTC,47727.638161,2026-08-31 08:00:00 UTC


In [185]:
wind  =wind.with_columns(
    pl.col("time")
    .str.to_datetime("%Y-%m-%d %H:%M:%S")
    .dt.replace_time_zone("UTC")
)
solar= solar.with_columns(
    pl.col("time")
    .str.to_datetime("%Y-%m-%d %H:%M:%S")
    .dt.replace_time_zone("UTC")
)

In [186]:
display(wind)

time,data
"datetime[μs, UTC]",i64
2026-01-01 00:00:00 UTC,42636
2026-01-01 00:15:00 UTC,43208
2026-01-01 00:30:00 UTC,43803
2026-01-01 00:45:00 UTC,44397
2026-01-01 01:00:00 UTC,44977
…,…
2026-08-31 22:45:00 UTC,30337
2026-08-31 23:00:00 UTC,30387
2026-08-31 23:15:00 UTC,30324


In [187]:
wind = wind.rename({"data": "wind"})
solar = solar.rename({"data": "solar"})

In [188]:


df= (
    df
    .join(wind, on="time", how="left")
    .join(solar, on="time", how="left")
    .sort("time")
)
display(df)

time,data,issued,wind,solar
"datetime[μs, UTC]",f64,"datetime[μs, UTC]",i64,i64
2026-01-01 00:00:00 UTC,39975.528491,2025-12-31 09:00:00 UTC,42636,0
2026-01-01 00:15:00 UTC,39478.928456,2025-12-31 09:00:00 UTC,43208,0
2026-01-01 00:30:00 UTC,39161.142551,2025-12-31 09:00:00 UTC,43803,0
2026-01-01 00:45:00 UTC,38936.679901,2025-12-31 09:00:00 UTC,44397,0
2026-01-01 01:00:00 UTC,38782.980144,2025-12-31 09:00:00 UTC,44977,0
…,…,…,…,…
2026-08-31 22:45:00 UTC,48236.690756,2026-08-31 08:00:00 UTC,30337,0
2026-08-31 23:00:00 UTC,47884.138047,2026-08-31 08:00:00 UTC,30387,0
2026-08-31 23:15:00 UTC,47727.638161,2026-08-31 08:00:00 UTC,30324,0


In [189]:
df= df.with_columns(
   (pl.col("data")- pl.col("wind")-pl.col("solar")).alias("residual load tech avail")
)



In [190]:
df.describe()

statistic,time,data,issued,wind,solar,residual load tech avail
str,str,f64,str,f64,f64,f64
"""count""","""25444""",25444.0,"""25444""",25444.0,25444.0,25444.0
"""null_count""","""0""",0.0,"""0""",0.0,0.0,0.0
"""mean""","""2026-04-26 09:08:01.834617+00:…",56148.997488,"""2026-04-25 07:15:50.793900+00:…",17088.562254,11580.589058,27479.846175
"""std""",null,9428.560732,null,12282.188025,16819.891067,18941.538798
"""min""","""2026-01-01 00:00:00+00:00""",34082.629982,"""2025-12-31 09:00:00+00:00""",268.0,0.0,-54246.696449
"""25%""","""2026-02-21 14:30:00+00:00""",48295.600499,"""2026-02-20 09:00:00+00:00""",7191.0,0.0,16505.839909
"""50%""","""2026-04-21 11:30:00+00:00""",55878.371017,"""2026-04-20 08:00:00+00:00""",14290.0,900.0,31075.887281
"""75%""","""2026-06-26 17:30:00+00:00""",64032.979906,"""2026-06-25 08:00:00+00:00""",24455.0,20214.0,41206.241819
"""max""","""2026-08-31 23:45:00+00:00""",78169.136587,"""2026-08-31 08:00:00+00:00""",61643.0,64530.0,66588.528346


In [191]:
df.write_parquet("tech_avail.parquet")